In [ ]:
# Make repaired datasets for all california jurisdictions
# SLOW!

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))
from data_repair import data_repair, _slugify

SUMMARY_FILEPATH = os.path.join(MY_DATA_PATH, f"dewey_summary.parquet")

COLUMNS = [
    'PERMIT_NUMBER', 'JURISDICTION', 'STATE', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'STATUS_NORMALIZED', 'STATUS_ORIGINAL', 'RECORD_TYPE_ORIGINAL', 'RECORD_SUBTYPE_ORIGINAL', 'APN', 'STREET', 'ZIPCODE', 'DATA', 'RESIDENTIAL'
]  # Columns to load from data file

OUTPUT_COLS = [k for k in COLUMNS if k != 'DATA'] # Don't include DATA in output cols to save space
OUTPUT_COLS = OUTPUT_COLS + ['STATUS_NORMALIZED_FLAG', 'FILE_DATE_FLAG', 'PERMIT_DATE_FLAG', 'FINAL_DATE_FLAG', 'INFERRED_SCHEMA']

OUTPUT_DIR = os.path.join(MY_DATA_PATH, "processed_data")

REPLACE = False   # Whether to replace existing output files


In [2]:
# Load the summary file
summ_df = pd.read_parquet(SUMMARY_FILEPATH)
summ_df = summ_df.loc[summ_df["STATE"] == "CA"]

In [3]:
# Jurisdiction / state
j_df = summ_df[['JURISDICTION', 'STATE']].drop_duplicates()
jurisdictions = j_df['JURISDICTION'].tolist()
states = j_df['STATE'].tolist()

In [ ]:
# Iterate through jurisdictions
t0 = time.time()
for i, (jurisdiction, state) in enumerate(zip(jurisdictions, states)):
    print(f"Processing {jurisdiction} {state} ({i+1}/{len(jurisdictions)})")

    jurisdiction_slug = _slugify(jurisdiction)
    state_slug = state.lower().strip()
    os.makedirs(os.path.join(OUTPUT_DIR, f"{state_slug}"), exist_ok=True)
    output_filepath = os.path.join(OUTPUT_DIR, f"{state_slug}", f"{state_slug}_{jurisdiction_slug}_repaired.parquet")

    if os.path.exists(output_filepath) and not REPLACE:
        print(f"    Skipping because output file already exists")
        continue

    files = summ_df.loc[(summ_df["JURISDICTION"] == jurisdiction) & (summ_df["STATE"] == state)]["FILENAME"].tolist()

    city_df = []
    for j, f in enumerate(files):
        dt = (time.time() - t0) / 60
        print(f"\r    Processing file {j+1}/{len(files)} ... elapsed time = {dt:.2f} minutes", end="", flush=True)
        temp_df = pd.read_parquet(os.path.join(DEWEY_PATH, f), columns=COLUMNS)
        temp_df = temp_df.loc[ (temp_df['JURISDICTION'] == jurisdiction) & (temp_df['STATE'] == state)].reset_index(drop=True)
        temp_df = data_repair(temp_df, jurisdiction=jurisdiction, state=state)
        city_df.append(temp_df[OUTPUT_COLS])
    city_df = pd.concat(city_df).reset_index(drop=True)
    city_df.to_parquet(output_filepath)
    print(f"\n    File saved to {output_filepath}.")

    if i>1:
        break


Processing Alameda CA (1/250)
    Processing file 11/11 ... elapsed time = 1.00 minutes
    File saved to /Users/ekung/Dropbox/projects/la-permits-data/processed_data/state_slug/ca_alameda_repaired.parquet.
Processing Alameda County CA (2/250)
    Processing file 2/7 ... elapsed time = 1.22 minutes

KeyError: 0

In [7]:
temp_df.loc[temp_df['PERMIT_NUMBER']=='BLD2021-00849'].iloc[0]['DATA']

'{"id": 1129608, "lat": null, "lng": null, "type": "Building Permit", "work": null, "closed": null, "issued": "2021-03-08 14:40:48Z", "search": {"id": 1129608, "type": "BLD", "msType": "Building Permit / BLD", "number": "BLD2021-00849", "status": "ISS - Issued", "typeId": 2, "address": "080 007802616", "dateVal": "2021-03-08T14:40:48Z", "msValue": "Project:1129608", "datePrefix": "Issued on", "createdDate": "2021-03-05T13:21:23Z", "description": "test", "projectType": "ProjectType:903", "projectMsValue": "Project:1129608", "statusBackColor": "Aqua", "canAddPublicComment": false}, "status": "ISS - Issued", "vlCode": null, "address": null, "created": "2021-03-05 13:21:23Z", "expired": "2022-03-12 00:00:00Z", "msValue": "Project:1129608", "isClosed": 0, "typeFull": "Building Permit / BLD", "addressId": null, "groupType": null, "humanName": "BLD2021-00849", "permitType": "BLD", "description": "test", "approxLocation": null}'